In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/data-d/validation_url.csv
/kaggle/input/data-d/test_url.csv
/kaggle/input/scienceqa-img-url/validation_url.csv
/kaggle/input/scienceqa-img-url/test_url.csv


# AppWrite

In [2]:
!pip install appwrite


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 1.8 MB/s eta 0:00:00


In [3]:
import os
import tempfile
import pandas as pd
from appwrite.client import Client
from appwrite.services.storage import Storage
from appwrite.input_file import InputFile


In [ ]:
PROJECT_ID = '##########'
BUCKET_ID = '#############'



# Initialize Appwrite client and storage service
client = Client()
client.set_endpoint('https://cloud.appwrite.io/v1')  # Your API Endpoint
client.set_project(PROJECT_ID)                       # Set your project ID
client.set_session('')                                 # The user session for authentication

storage = Storage(client)

In [5]:
#import ast

def upload_image(row):
    image = row['image']
    try:
        # Save the PIL image to a temporary file in PNG format.
        with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as tmp_file:
            temp_filename = tmp_file.name
            image.save(temp_filename)
        
        # Upload the image using Appwrite Storage service.
        result = storage.create_file(
            bucket_id=BUCKET_ID,  # Replace with your bucket ID
            file_id='unique()',                 # Generate a unique file ID
            file=InputFile.from_path(temp_filename),
            permissions=["read(\"any\")"]       # Optional permissions
        )
        
        # Construct the image URL.
        uploaded_file_id = result['$id']
        print(result)
        image_url = f"{client._endpoint}/storage/buckets/{BUCKET_ID}/files/{uploaded_file_id}/view?project={PROJECT_ID}&mode=admin"
        
        
        
        # Clean up the temporary file.
        os.remove(temp_filename)
        row['image_url'] = image_url
        print("done")
        
    
    except Exception as e:
        # Print error for debugging and return None if upload fails.
        print(f"Upload failed for image: {e}")
        row['image_url'] = None
    return row


In [ ]:
# s = sample.map(upload_image)
# s


In [7]:
!pip install datasets -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.12.0 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.8.4.1 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cudnn-cu12==9.1.0.70; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cudnn-cu12 9.3.0.75 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cufft-cu12==11.2.1.3; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cufft-cu12 11.3.3.83 which is incompatible.
torch 2.5.1+cu124 requires nvidia-curand-cu12==10.3.5.147; platform_system == "Linux" and platform_machin

In [8]:
from datasets import load_dataset
# dataset = load_dataset("/kaggle/input/data-d")

# already done

In [9]:
image_with_url = load_dataset("/kaggle/input/scienceqa-img-url")
image_with_url_validation = image_with_url['validation']
image_with_url_test = image_with_url['test']

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [10]:
validation_img_index_no_urls = []
for i, row in enumerate(image_with_url_validation):
    if row['image_url'] == None:
        validation_img_index_no_urls.append(i)

In [11]:
print(validation_img_index_no_urls)

[86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 470, 471, 472, 473, 474, 475, 476, 477, 478, 479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 549, 550, 551, 552, 553, 554, 555, 556, 557, 558, 559, 560, 561, 562, 563, 564, 625, 626, 627, 628, 629, 630, 631, 632, 633, 634, 635, 636, 637, 638, 639, 640, 641, 642, 643, 644, 645, 646, 647, 648, 649, 710, 711, 712, 713, 714, 715, 716, 717, 718, 719, 720, 721, 722, 723, 724, 725, 726, 727, 728, 729, 730, 791, 792, 793, 794, 795, 796, 797, 798, 799, 800, 801, 802, 803, 804, 805, 806, 807, 808, 809, 810, 811, 872, 873, 874, 875, 876, 877, 878, 879, 880, 881, 882, 883, 884, 885, 886, 887, 888

In [12]:
test_img_index_no_urls = []
for i, row in enumerate(image_with_url_test):
    if row['image_url'] == None:
        test_img_index_no_urls.append(i)

In [13]:
print(test_img_index_no_urls)

[3, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 296, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 427, 428, 429, 430, 431, 432, 493, 494, 495, 496, 497, 498, 499, 500, 501, 502, 503, 504, 505, 506, 507, 508, 509, 51

# original

In [14]:
#import pandas as pd

scienceqa = load_dataset("lmms-lab/ScienceQA-IMG")
#qa_test_images = pd.DataFrame(scienceqa["test"]["image"])
#qa_validation_images = pd.DataFrame(scienceqa["validation"]["image"])
test_science = scienceqa["test"] ## This is a formatted and filtered version of derek-thomas/ScienceQA with only image instances
validation_science = scienceqa["validation"]
print(test_science)
print(validation_science)

README.md:   0%|          | 0.00/2.06k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/400M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/133M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/130M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6218 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2097 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2017 [00:00<?, ? examples/s]

Dataset({
    features: ['image', 'question', 'choices', 'answer', 'hint', 'task', 'grade', 'subject', 'topic', 'category', 'skill', 'lecture', 'solution'],
    num_rows: 2017
})
Dataset({
    features: ['image', 'question', 'choices', 'answer', 'hint', 'task', 'grade', 'subject', 'topic', 'category', 'skill', 'lecture', 'solution'],
    num_rows: 2097
})


In [15]:
for i in test_img_index_no_urls[0:2]:
    print(test_science[i]['question'], end="\n\n")


What is the expected ratio of offspring with a woolly fleece to offspring with a hairy fleece? Choose the most likely ratio.

What is the capital of Kansas?



In [16]:
for i in test_img_index_no_urls[-2:]:
    print(image_with_url_test[i]['question'], end="\n\n")
    

Which better describes the Pantanal ecosystem?

Which continent is highlighted?



# now follow this index and upload those images

In [17]:
#import pandas as pd
# Update missing images in the test set
import traceback

def fill_missing_images_val(example, idx):
    if idx in validation_img_index_no_urls:
        example = upload_image(example)
        
    return example

# test_science = test_science.map(fill_missing_images, with_indices=True)
# validation_science = validation_science.map(fill_missing_images, with_indices=True)
#test_science = test_science.filter(lambda x: x["image_url"] == None)
#validation_science = validation_science.filter(lambda x: x["image_url"] == None)
#test_science = test_science.to_pandas()
#validation_science = validation_science.to_pandas()

#test_science = test_science.merge(qa_test_images, left_on= "image", right_on= 0)


# sample = ScienceQA['train'].select(range(0,1))
# sample

In [18]:
def fill_missing_images_test(example, idx):
    if idx in test_img_index_no_urls:
        example = upload_image(example)
        
    return example

# delete all from the cloud APPwrite and test with a small sample then run with whole code

In [19]:
sample = test_science.select(range(3,4))

In [20]:
sample = sample.map(fill_missing_images_test, with_indices=True)

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

# change database settings to write and use yours I've bloked mine so that they don't get mixed up

# while saving save both one with image field, and one without it

In [21]:
sample.to_csv("sample.csv")

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

9377

# save the ruturned dataset so we will have two split of each

In [22]:
# test_science.to_csv("test_url_final.csv")
#ScienceQA['validation'] = ScienceQA['validation'].map(upload_image)

In [23]:
# validation_science = validation_science.map(upload_image)
# #ScienceQA['validation'].to_csv("validation_url.csv")

In [24]:
# validation_science.to_csv("validation_url_final.csv")
# #cienceQA['test'] = ScienceQA['test'].map(upload_image)

In [25]:
#ScienceQA['test'].to_csv("test_url.csv")

In [26]:
#for split in ScienceQA.keys():
    #ScienceQA[split] = ScienceQA[split].remove_columns(["image"])

In [27]:
#ScienceQA['validation'].to_csv("validation.csv")

In [28]:
#ScienceQA['test'].to_csv("test.csv")

# Check dataset

In [29]:
# # Check each split for missing images
# for split in ScienceQA.keys():
#     missing_images = ScienceQA[split].filter(lambda row: row['image'] is None)
#     if len(missing_images) > 0:
#         print(f"Split '{split}' has {len(missing_images)} rows with missing images.")
#         # for row in missing_images:
#         #     print(row)
#     else:
#         print(f"All rows in split '{split}' have an image.")

# Image to url

In [30]:
# !pip3 install cloudinary -q

In [ ]:
# import cloudinary
# import cloudinary.uploader
# from cloudinary.utils import cloudinary_url

# # Configuration       
# cloudinary.config( 
#     cloud_name = "drv7kjkna", 
#     api_key = "################", 
#     api_secret = "######################", # Click 'View API Keys' above to copy your API secret
#     secure=True
# )

## Need to upload all image and store the url

In [32]:
# import io
# import cloudinary.uploader
# def upload_image(row):
#     img_buffer = io.BytesIO()
#     row["image"].save(img_buffer, format="PNG")
#     img_buffer.seek(0) 
    
#     upload_result = cloudinary.uploader.upload(img_buffer)
#     row["image_url"] = upload_result["url"]
#     return row

In [33]:
# for split in ScienceQA.keys():
#     ScienceQA[split] = ScienceQA[split].map(upload_image, load_from_cache_file=False)

In [34]:
# ScienceQA['test'] = ScienceQA['test'].map(upload_image, load_from_cache_file=False)

In [35]:
# for split in ScienceQA.keys():
#     ScienceQA[split] = ScienceQA[split].remove_columns(["image"])

In [36]:
# ScienceQA['test'] = ScienceQA['test'].remove_columns(["image"])

In [37]:
# print("After:")
# print(ScienceQA)

In [38]:
# # Save the train and test splits as CSV files
# # ScienceQA["train"].to_csv("train.csv")
# ScienceQA["test"].to_csv("test.csv")
# # ScienceQA["validation "].to_csv("validation.csv")


Groq

In [39]:
# !pip install groq -q

In [40]:
# from groq import Groq

Testing

In [41]:
# test_q = ScienceQA['train'][0]
# test_q

In [42]:
# import io
# import cloudinary.uploader

# # Convert the PIL image to a bytes buffer
# img_buffer = io.BytesIO()
# test_q["image"].save(img_buffer, format="PNG")
# img_buffer.seek(0)  # Reset the stream position to the beginning

# # Upload the image using the bytes stream
# upload_result = cloudinary.uploader.upload(img_buffer)



In [43]:
# print(upload_result["url"])

In [ ]:
# GROQ_API_KEY="####################"

In [45]:
# os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [46]:
# client = Groq(
#     api_key=os.environ.get("GROQ_API_KEY"),
# )
# completion = client.chat.completions.create(
#     model="llama-3.2-11b-vision-preview",
#     messages=[
#         {
#             "role": "user",
#             "content": [
#                  {
#                     "type": "text",
#                     "text": test_q["question"]
#                 },
#                 {
#                     "type": "text",
#                     "text": f"Hint: {test_q['hint']}"
#                 },
#                 {
#                     "type": "text",
#                     "text": "Choices:"
#                 },
#                 {
#                     "type": "text",
#                     "text": "\n".join(test_q["choices"])
#                 },
#                 {
#                     "type": "image_url",
#                     "image_url": {
#                         "url": upload_result["url"]
#                     }
#                 }
#             ]
#         }
#     ],
#     temperature=1,
#     max_completion_tokens=1024,
#     top_p=1,
#     stream=False,
#     stop=None,
# )

# print(completion.choices[0].message)


In [47]:
# test_q['choices'][test_q["answer"]]